# Week 24, Day 2: pytest

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w024d2.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
print(0.1 + 0.2 == 0.3)
print(round(0.1 + 0.2, 10) == round(0.3, 10))

In [ ]:
import pytest

with open("shop.py", "w") as f:
    f.write("def total(prices, discount=0):\n    return sum(prices) - discount\n")

with open("test_shop.py", "w") as f:
    f.write('''
from shop import total

def test_no_discount():
    assert total([2, 3]) == 5

def test_discount_is_a_fraction():
    assert total([10, 10], discount=0.25) == 15
''')

pytest.main(["-q", "-p", "no:cacheprovider", "test_shop.py"])

In [ ]:
import pytest

def withdraw(balance, amount):
    if amount <= 0:
        raise ValueError("amount must be positive")
    if amount > balance:
        raise ValueError("not enough money")
    return balance - amount

def test_overdraw():
    with pytest.raises(ValueError):
        withdraw(10, 50)

def test_negative_message():
    with pytest.raises(ValueError, match="positive"):
        withdraw(10, -5)

def test_error_details():
    with pytest.raises(ValueError) as info:
        withdraw(10, 50)
    assert "not enough" in str(info.value)

test_overdraw(); test_negative_message(); test_error_details()
print("all passed")

In [ ]:
import pytest

print(0.1 + 0.2 == pytest.approx(0.3))
print([0.1 + 0.2, 1 / 3] == pytest.approx([0.3, 0.3333333]))
print(9.9 == pytest.approx(10, abs=0.2))        # an absolute tolerance
print(9.9 == pytest.approx(10))

In [ ]:
import pytest

with open("test_words.py", "w") as f:
    f.write('''
import pytest

def shout(text):
    return text.upper() + "!"

@pytest.mark.parametrize("text, expected", [
    ("hi", "HI!"),
    ("", "!"),
    ("ok then", "OK THEN!"),
    ("ça va", "ÇA VA!"),
])
def test_shout(text, expected):
    assert shout(text) == expected
''')

pytest.main(["-v", "-p", "no:cacheprovider", "test_words.py"])

> **Key idea.** pytest finds `test_*.py` files and runs their `test_` functions. When an `assert` fails it shows the values involved. Use `pytest.raises(Error, match=...)` to check errors, `pytest.approx` to compare floats, and `@pytest.mark.parametrize` to run one test over many cases. `-v`, `-x`, `-k` and `--lf` control what runs and how much you see.

### Exercise: Parametrize is_leap

Write **one** parametrized test, `test_is_leap`, for `is_leap` in `calendar_rules.py`, with at least four cases, that catches every broken version.

In [ ]:
# Run this cell first: it creates the files used below.
with open('calendar_rules.py', 'w', encoding='utf-8') as f:
    f.write('def is_leap(year):\n    return year % 4 == 0 and (year % 100 != 0 or year % 400 == 0)\n')

In [ ]:
import pytest
from calendar_rules import is_leap

@pytest.mark.parametrize("year, expected", [
    (2024, True),
    ___
])
def test_is_leap(year, expected):
    assert is_leap(year) == expected

In [ ]:
# Check: run the exercise cell above first, then this one.
check("GOOD = open(\"calendar_rules.py\").read()\nBROKEN = {\n    \"every fourth year\": \"def is_leap(y):\\n    return y % 4 == 0\\n\",\n    \"forgets 400\": \"def is_leap(y):\\n    return y % 4 == 0 and y % 100 != 0\\n\",\n    \"always true\": \"def is_leap(y):\\n    return True\\n\",\n}\ndef test_param():\n    \"one parametrized test with at least four cases\"\n    assert \"parametrize\" in source, \"use @pytest.mark.parametrize\"\n    r = pytest_run(source)\n    assert len(r[\"passed\"]) >= 4 and all(p.startswith(\"test_is_leap[\") for p in r[\"passed\"]), \\\n        \"test_is_leap should run at least four cases, all passing\"\ndef test_catches():\n    \"it catches every broken version\"\n    results = against(source, \"calendar_rules\", BROKEN)\n    write_file(\"calendar_rules.py\", GOOD)\n    missed = [k for k, v in results.items() if not v]\n    assert not missed, f\"not caught: {missed}\"")

### Exercise: Test the errors

`withdraw(balance, amount)` in `bank.py` raises `ValueError("amount must be positive")` for amounts of zero or less, and `ValueError("not enough money")` for amounts over the balance. Write tests with `pytest.raises` (using `match`) for both errors, and a test that a normal withdrawal works, so that every broken version is caught.

In [ ]:
# Run this cell first: it creates the files used below.
with open('bank.py', 'w', encoding='utf-8') as f:
    f.write('def withdraw(balance, amount):\n    if amount <= 0:\n        raise ValueError("amount must be positive")\n    if amount > balance:\n        raise ValueError("not enough money")\n    return balance - amount\n')

In [ ]:
import pytest
from bank import withdraw

In [ ]:
# Check: run the exercise cell above first, then this one.
check("GOOD = open(\"bank.py\").read()\nBROKEN = {\n    \"zero allowed\": GOOD.replace(\"amount <= 0\", \"amount < 0\"),\n    \"can't withdraw everything\": GOOD.replace(\"amount > balance\", \"amount >= balance\"),\n    \"no overdraw check\": GOOD.replace('    if amount > balance:\\n        raise ValueError(\"not enough money\")\\n', \"\"),\n    \"wrong error type\": GOOD.replace('raise ValueError(\"not enough money\")', 'raise RuntimeError(\"not enough money\")'),\n    \"messages swapped\": GOOD.replace(\"amount must be positive\", \"TMP\").replace(\"not enough money\", \"amount must be positive\").replace(\"TMP\", \"not enough money\"),\n    \"adds instead\": GOOD.replace(\"balance - amount\", \"balance + amount\"),\n}\ndef test_style():\n    \"uses pytest.raises with match\"\n    assert \"pytest.raises\" in source and \"match=\" in source\ndef test_catches():\n    \"your tests pass on the correct code and catch every broken version\"\n    results = against(source, \"bank\", {\"correct\": GOOD, **BROKEN})\n    write_file(\"bank.py\", GOOD)\n    assert results[\"correct\"] == [], f\"these fail on the correct code: {results['correct']}\"\n    missed = [k for k in BROKEN if not results[k]]\n    assert not missed, f\"not caught: {missed}\"")

### Exercise: Floats, approximately

`mean_temperature(readings)` in `weather.py` returns the mean of a list of temperatures, rounded to 2 decimal places, and `to_fahrenheit(c)` converts Celsius to Fahrenheit (`c * 9 / 5 + 32`). Write tests that use `pytest.approx` and catch every broken version.

In [ ]:
# Run this cell first: it creates the files used below.
with open('weather.py', 'w', encoding='utf-8') as f:
    f.write('def mean_temperature(readings):\n    return round(sum(readings) / len(readings), 2)\n\ndef to_fahrenheit(c):\n    return c * 9 / 5 + 32\n')

In [ ]:
import pytest
from weather import mean_temperature, to_fahrenheit

In [ ]:
# Check: run the exercise cell above first, then this one.
check("GOOD = open(\"weather.py\").read()\nBROKEN = {\n    \"rounds to 1 place\": GOOD.replace(\"round(sum(readings) / len(readings), 2)\", \"round(sum(readings) / len(readings), 1)\"),\n    \"floor division\": GOOD.replace(\"sum(readings) / len(readings)\", \"sum(readings) // len(readings)\"),\n    \"forgets the 32\": GOOD.replace(\"c * 9 / 5 + 32\", \"c * 9 / 5\"),\n    \"9 / 5 the wrong way\": GOOD.replace(\"c * 9 / 5 + 32\", \"c * 5 / 9 + 32\"),\n}\ndef test_style():\n    \"uses pytest.approx\"\n    assert \"approx\" in source\ndef test_catches():\n    \"your tests pass on the correct code and catch every broken version\"\n    results = against(source, \"weather\", {\"correct\": GOOD, **BROKEN})\n    write_file(\"weather.py\", GOOD)\n    assert results[\"correct\"] == [], f\"these fail on the correct code: {results['correct']}\"\n    missed = [k for k in BROKEN if not results[k]]\n    assert not missed, f\"not caught: {missed}\"")

### Exercise: Read the report

A pytest run printed this. Answer from it.

```
..F.F                                                    [100%]
=========================== FAILURES ===========================
___________________ test_total[prices2-12.5] ___________________

prices = [5, 7.5], expected = 12.5

    @pytest.mark.parametrize("prices, expected", CASES)
    def test_total(prices, expected):
>       assert total(prices) == expected
E       assert 12 == 12.5
E        +  where 12 = total([5, 7.5])

test_cart.py:9: AssertionError
_________________________ test_empty ___________________________

    def test_empty():
>       assert total([]) == 0
E       TypeError: 'NoneType' object is not iterable

test_cart.py:12: TypeError
=================== 2 failed, 3 passed in 0.03s ================
```

In [ ]:
a = ___   # how many tests passed?
b = ___   # what did total([5, 7.5]) return?
c = ___   # the name of the test that crashed with an error, rather than a failed comparison
d = ___   # True or False: the report shows every case of test_total failed

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert a == 3 and b == 12 and c == \"test_empty\" and d is False")

### Exercise: Pick the command

Write the full command for each, as a string.

In [ ]:
a = ___   # run all tests, stopping at the first failure
b = ___   # run only tests whose names contain "refund"
c = ___   # rerun just the tests that failed last time
d = ___   # run only the test test_empty in the file tests/test_cart.py

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert a == \"pytest -x\" and b in (\"pytest -k refund\", 'pytest -k \"refund\"') and c == \"pytest --lf\"\n    assert d == \"pytest tests/test_cart.py::test_empty\"")

### Bug hunt: Never collected

These tests look fine, but pytest reports "no tests ran". Fix them so pytest finds and runs both.

In [ ]:
def double(x):
    return x * 2

def check_double():
    assert double(3) == 6

def double_zero_test():
    assert double(0) == 0

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_collected():\n    \"pytest collects and passes two tests\"\n    r = pytest_run(source)\n    assert len(r[\"passed\"]) == 2, f\"pytest ran: {r['passed'] + r['failed']}\"")

### Bug hunt: Right code, failing test

`split_bill(150, 4)` returns 37.5 and `split_bill(100, 3)` returns 33.333..., which is right, but the test fails. Fix the test (not the function), keeping it a real check.

In [ ]:
import pytest

def split_bill(total, people):
    return total / people

def test_split():
    assert split_bill(150, 4) == 37.5
    assert split_bill(100, 3) == 33.33

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_passes():\n    \"the test passes, and still catches a wrong split\"\n    r = pytest_run(source)\n    assert r[\"failed\"] == [] and r[\"passed\"], \"the test still fails\"\n    import ast\n    tests_only = \"import pytest\\nfrom bill import split_bill\\n\" + \"\\n\".join(\n        ast.unparse(n) for n in ast.parse(source).body if isinstance(n, ast.FunctionDef) and n.name.startswith(\"test_\"))\n    res = against(tests_only, \"bill\", {\"bad\": \"def split_bill(total, people):\\n    return round(total / people)\\n\"})\n    assert res[\"bad\"], \"the test no longer catches a split that rounds to whole pounds\"")

### Bug hunt: The check that never runs

This test passes, but the `assert` inside the `with` block never runs. Move it so it does, and fix it so it checks the right message.

In [ ]:
import pytest

def parse_age(text):
    age = int(text)
    if age < 0:
        raise ValueError("age can't be negative")
    return age

def test_negative():
    with pytest.raises(ValueError) as info:
        parse_age("-3")
        assert "positive" in str(info.value)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_runs():\n    \"the assert is after the with block, and the test passes\"\n    import ast\n    tree = ast.parse(source)\n    fn = next(n for n in tree.body if isinstance(n, ast.FunctionDef) and n.name == \"test_negative\")\n    w = next(n for n in fn.body if isinstance(n, ast.With))\n    assert not any(isinstance(n, ast.Assert) for n in w.body), \"the assert is still inside the with block\"\n    assert any(isinstance(n, ast.Assert) for n in fn.body), \"put the assert after the with block\"\n    r = pytest_run(source)\n    assert r[\"failed\"] == [] and r[\"passed\"] == [\"test_negative\"], \"the test should pass\"")

### Puzzle: A parametrized test

Arrange the lines. One line is a trap.

(Put these lines in order on the website.)

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: test a duration parser

`parse_duration(text)` in `durations.py` turns text like `"1h30m"`, `"45m"`, `"2h"` or `"90s"` into a number of seconds. Hours, minutes and seconds can appear in any combination, but always in that order (`"1h2m3s"`). Spaces and capital letters aren't allowed. Empty text, or anything else, raises `ValueError`. Write tests, using parametrize and `pytest.raises`, that pass on the correct code and catch all **six** broken versions.

In [ ]:
# Run this cell first: it creates the files used below.
with open('durations.py', 'w', encoding='utf-8') as f:
    f.write('import re\n\ndef parse_duration(text):\n    m = re.fullmatch(r"(?:(\\d+)h)?(?:(\\d+)m)?(?:(\\d+)s)?", text)\n    if not text or m is None:\n        raise ValueError(f"not a duration: {text!r}")\n    h, mins, s = (int(g) if g else 0 for g in m.groups())\n    return h * 3600 + mins * 60 + s\n')

In [ ]:
import pytest
from durations import parse_duration

In [ ]:
# Check: run the exercise cell above first, then this one.
check("GOOD = open(\"durations.py\").read()\nBROKEN = {\n    \"minutes as 100\": GOOD.replace(\"mins * 60\", \"mins * 100\"),\n    \"empty gives 0\": GOOD.replace(\"if not text or m is None\", \"if m is None\"),\n    \"seconds ignored\": GOOD.replace(\"+ mins * 60 + s\", \"+ mins * 60\"),\n    \"search not fullmatch\": GOOD.replace(\"re.fullmatch(\", \"re.match(\"),\n    \"case ignored\": GOOD.replace(', text)\\n    if not', ', text, re.I)\\n    if not'),\n    \"hours as 60\": GOOD.replace(\"h * 3600\", \"h * 60\"),\n}\ndef test_style():\n    \"uses parametrize and pytest.raises\"\n    assert \"parametrize\" in source and \"pytest.raises\" in source\ndef test_catches():\n    \"your tests pass on the correct code and catch all six broken versions\"\n    results = against(source, \"durations\", {\"correct\": GOOD, **BROKEN})\n    write_file(\"durations.py\", GOOD)\n    assert results[\"correct\"] == [], f\"these fail on the correct code: {results['correct']}\"\n    missed = [k for k in BROKEN if not results[k]]\n    assert not missed, f\"not caught: {missed}\"")

## Warm-up

Floats are stored in binary, so 0.1 + 0.2 comes out as 0.30000000000000004. A test that compares floats with `==` can fail even when the code is right. pytest has a tidy answer to this, `pytest.approx`, which you'll meet today.

## Running pytest

pytest is installed with `pip install pytest` (here it's ready to use). In a project, you run it from the project folder in a terminal:

```
$ pytest
```

It then:

1. looks in the folder (and subfolders) for files named `test_*.py` or `*_test.py`;
2. in each, collects functions whose names start with `test_`, and methods starting with `test_` in classes whose names start with `Test`;
3. runs every one, and reports.

A typical project keeps tests in their own folder:

```
budget/
├── budget.py
└── tests/
    ├── test_totals.py
    └── test_parsing.py
```

Each test file imports what it tests (`from budget import monthly_total`) and contains plain `test_` functions with `assert`, exactly like yesterday's. No special classes, no `self.assertEqual`: that's a big part of why pytest is so popular. (Python also comes with a test module called `unittest`, which works in a heavier, class-based style; you'll see it in older code, and pytest runs those tests too.)

Useful options:

| Command | What it does |
|---|---|
| `pytest -v` | verbose: one line per test, with its name |
| `pytest -q` | quiet: just dots and a summary |
| `pytest -x` | stop at the first failure |
| `pytest -k leap` | run only tests whose names contain "leap" |
| `pytest tests/test_totals.py` | run one file |
| `pytest tests/test_totals.py::test_empty` | run one test |
| `pytest --lf` | rerun only the tests that failed last time |

## Reading a report

Here's pytest run from inside Python, on a test file written for the example. (In a terminal you'd just type `pytest -q`.)

Read the report from the bottom up:

- the **summary line** says how many passed and failed;
- each failure has a section headed with the test's name;
- the failing line is marked with `>`, and the `E` lines show **the values** involved: `assert 19.75 == 15`, where 19.75 came from `total([10, 10], discount=0.25)`.

That last part is pytest's best feature. It rewrites your `assert` statements so that when one fails, it can show what each side was. You don't need messages like `f"expected 15, got {result}"`; pytest prints that for you.

Here the test describes the intended behaviour (a discount of 0.25 means 25% off), and the code subtracts 0.25 instead. The report tells you the code is wrong. Sometimes, of course, it's the test that's wrong; the report gives you the values to decide.

## Errors and floats

**Testing that an error is raised.** Yesterday's `try`/`except`/`assert False` pattern works, but pytest has a clearer one:

- `with pytest.raises(ValueError):` passes only if the code inside raises `ValueError` (or a subclass). If nothing is raised, the test fails with "DID NOT RAISE".
- `match="..."` also checks the message; it's a regular expression searched in the message, so plain words work.
- `as info` keeps the exception, in `info.value`, for more checks after the `with` block.

Keep only the call that should raise inside the `with`. Any line after it in the block never runs, because the error jumps straight out.

**Comparing floats.** `pytest.approx` compares within a small tolerance (by default, about one part in a million):

Use `approx` whenever a float comes out of arithmetic. Whole numbers and strings can still use plain `==`.

## Parametrize

Testing one function on many inputs leads to lots of nearly identical tests. **Parametrize** writes one test and gives it a list of cases:

```
import pytest
from calendar_rules import is_leap

@pytest.mark.parametrize("year, expected", [
    (2024, True),
    (2023, False),
    (1900, False),
    (2000, True),
])
def test_is_leap(year, expected):
    assert is_leap(year) == expected
```

- The first argument names the parameters, as one string separated by commas.
- The second is a list of cases; each tuple gives those parameters' values.
- pytest runs the test once per case, and reports each separately: `test_is_leap[2024-True]`, `test_is_leap[1900-False]` and so on. When one fails, the name says which case.

A single parameter needs no tuples: `@pytest.mark.parametrize("word", ["", " ", "\t"])`.

Parametrize is the right tool when the **same check** applies to many inputs. When the checks differ, write separate tests, so each name says what it checks.

## Skipping and expected failures

Two marks you'll see in real projects:

- `@pytest.mark.skip(reason="...")` skips a test, and `@pytest.mark.skipif(condition, reason="...")` skips it when the condition is true (only on Windows, say).
- `@pytest.mark.xfail(reason="...")` marks a test as **expected to fail**: a known bug not yet fixed. It's reported as "xfail" rather than a failure, and as "XPASS" if it unexpectedly passes.

Use them sparingly. A skipped test checks nothing, and a project full of skips is quietly losing its safety net.

## Practice

In these exercises your code is a test file. The checks run it with real pytest: against the correct code, where every test must pass, and against broken versions, which your tests must catch.

## Quiz

## Daily challenge

## Before you go

If you have Python on your own computer, `pip install pytest`, put yesterday's tests in a `test_something.py` file next to the code, and run `pytest -v` in that folder. Tomorrow: designing good tests, so they stay useful as the code grows.